In [ ]:
! unzip /kaggle/input/ocrdataset.zip

In [ ]:
import torch.nn as nn
import torch

class Convmodule(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 64, kernel_size=(3,3), stride=(1,1), padding=1)
        self.relu1 = nn.ReLU(inplace=True)
        self.pool1 = nn.MaxPool2d(kernel_size=(2,2), stride=(2,2))

        self.conv2 = nn.Conv2d(64, 128, kernel_size=(3,3), stride=(1,1), padding=1)
        self.relu2 = nn.ReLU(inplace=True)
        self.pool2 = nn.MaxPool2d(kernel_size=(2,2), stride=(2,2))

        self.conv3 = nn.Conv2d(128, 256, kernel_size=(3,3), stride=(1,1), padding=1)
        self.relu3 = nn.ReLU(inplace=True)
        self.conv4 = nn.Conv2d(256, 256, kernel_size=(3,3), stride=(1,1), padding=1)
        self.relu4 = nn.ReLU(inplace=True)


        self.pool4 = nn.MaxPool2d(kernel_size=(2,2), stride=(2,1), padding=(0,1))

        self.conv5 = nn.Conv2d(256, 512, kernel_size=(3,3), stride=(1,1), padding=1)
        self.batchnorm1 = nn.BatchNorm2d(512)
        self.relu5 = nn.ReLU(inplace=True)

        self.conv6 = nn.Conv2d(512, 512, kernel_size=(3,3), stride=(1,1), padding=1)
        self.batchnorm2 = nn.BatchNorm2d(512)
        self.relu6 = nn.ReLU(inplace=True)

        self.pool6 = nn.MaxPool2d(kernel_size=(2,2), stride=(2,1), padding=(0,1))

        self.conv7 = nn.Conv2d(512, 512, kernel_size=(2,2), stride=(1,1), padding=0)

    def forward(self, input):
        x = self.pool1(self.relu1(self.conv1(input)))
        x = self.pool2(self.relu2(self.conv2(x)))

        x = self.relu3(self.conv3(x))
        x = self.pool4(self.relu4(self.conv4(x)))

        x = self.batchnorm1(self.conv5(x))
        x = self.relu5(x)

        x = self.batchnorm2(self.conv6(x))
        x = self.relu6(x)
        x = self.pool6(x)

        x = self.conv7(x)
        return x


class CRNN(nn.Module):
    def __init__(self, num_labels, hidden_size):
        super(CRNN, self).__init__()

        self.cnn = Convmodule()

        self.rnn = nn.LSTM(
            input_size=512,
            hidden_size=hidden_size,
            num_layers=2,
            bidirectional=True,
            batch_first=True
        )


        self.fc = nn.Linear(hidden_size * 2, num_labels + 1) #numlables +1 for blank tokens

    def forward(self, x):
        features = self.cnn(x)

        B, C, H, W = features.size() #=> [B,512,1,seq]

        features = features.squeeze(2) #=>[B,512,seq]

        features = features.permute(0,2,1) # Shape: [B, SeqLen, 512]
        rnn_out, _ = self.rnn(features) # Shape: [B, SeqLen, 512]

        output = self.fc(rnn_out) # Shape: [B, SeqLen, 82]

        return output


In [ ]:
import os
import pandas as pd

SUBSETS = ["/kaggle/input/ocrdatasettask2/easy", "/kaggle/input/ocrdatasettask2/hard", "/kaggle/input/ocrdatasettask2/bonus"]

def get_vocabulary(subsets):
    unique_chars = set()

    for subset in subsets:
        csv_path = os.path.join(subset, "labels.csv")



        df = pd.read_csv(csv_path, header=None, names=["filename", "label","datatype"], dtype=str)


            # Join all labels into one massive string and get unique characters
        all_text = "".join(df["label"].tolist())
        unique_chars.update(set(all_text))

        print(f"  -> Processed {subset}: Found {len(df)} labels.")



    sorted_vocab = sorted(list(unique_chars))
    return "".join(sorted_vocab)

VOCAB = get_vocabulary(SUBSETS)


numclass = len(VOCAB)
print(f"String: '{VOCAB}'")
print(f"Length: {numclass} characters")

In [ ]:
class LabelEncoderDecoder():

  def __init__(self,vocab):
    self.vocab = vocab
    self.char2int = {char: i + 1 for i, char in enumerate(vocab)}
    self.int2char = {i + 1: char for i, char in enumerate(vocab)}
    self.blank_idx = 0
  def encode(self,text):
    encoded = [self.char2int[char] for char in text]
    return encoded

  def decode(self,encoded):
    decoded = "".join([self.int2char[char] for char in encoded if char != 0])
    return decoded

In [ ]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from PIL import Image
encoder = LabelEncoderDecoder(VOCAB)

class CaptchDataset(Dataset):

  def __init__(self,folder,labels,transform =  None):
    self.imgdir = folder
    self.labels = labels

    self.transform = transform
    self.df = pd.read_csv(self.labels,header=None,names=["filename","label","imagetype"],dtype=str)

  def __len__(self):
    return len(self.df)

  def __getitem__(self,idx):
    row = self.df.iloc[idx]
    img_path = os.path.join(self.imgdir, row["filename"])
    label_text = str(row["label"])

    image = Image.open(img_path).convert("RGB")
    imagetype = str(row["imagetype"])

    if self.transform:
          image = self.transform(image)

    # Encode label to list then to a tensor
    label_seq = torch.tensor(encoder.encode(label_text), dtype=torch.long)
    return image, label_seq, len(label_seq) , imagetype



In [ ]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((32, 128)), # Fixed size for batching
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [ ]:
EASY = CaptchDataset("/kaggle/input/ocrdatasettask2/easy",labels="/kaggle/input/ocrdatasettask2/easy/labels.csv",transform=transform)
HARD = CaptchDataset("/kaggle/input/ocrdatasettask2/hard",labels="/kaggle/input/ocrdatasettask2/hard/labels.csv",transform = transform)
BONUS = CaptchDataset("/kaggle/input/ocrdatasettask2/bonus",labels="/kaggle/input/ocrdatasettask2/bonus/labels.csv",transform = transform)

In [ ]:
import matplotlib.pyplot as plt

datasets = [("Easy Set", EASY), ("Hard Set", HARD), ("Bonus Set", BONUS)]
fig, axes = plt.subplots(3, 2, figsize=(12, 8))

for row_idx, (name, dataset) in enumerate(datasets):
    indices = torch.randint(0, len(dataset), (2,))

    for col_idx, idx in enumerate(indices):
        img_tensor, label_seq, label_len,_ = dataset[idx.item()]

        # Convert label sequence back to text using your 'encoder'
        if torch.is_tensor(label_seq):
            label_seq = label_seq.tolist()
        decoded_text = encoder.decode(label_seq)

        # Plotting
        ax = axes[row_idx, col_idx]
        ax.imshow(img_tensor.permute(1, 2, 0))
        ax.set_title(f"{name}\nLabel: {decoded_text}")
        ax.axis('off')


In [ ]:
from torch.utils.data import ConcatDataset, Subset
from torch.utils.data import random_split

train_ratio = 0.85

red_indices = [i for i, row in BONUS.df.iterrows() if "red" == row["imagetype"]]
green_indices = [i for i, row in BONUS.df.iterrows() if "green" == row["imagetype"]]

BONUS_RED = Subset(BONUS, red_indices)
BONUS_GREEN = Subset(BONUS, green_indices)



# split each dataset with 0.85 and 0.15 and concatente everything
def split_it(dataset):
  DATASET_SIZE = len(dataset)
  train_size = int(DATASET_SIZE * 0.85)
  validation = DATASET_SIZE - train_size
  train_dataset, valid_dataset = random_split(dataset, [train_size, validation])

  return train_dataset , valid_dataset


Train_Bonusred , Valid_Bonusred = split_it(BONUS_RED)
Train_Bonusgreen , Valid_Bonusgreen = split_it(BONUS_GREEN)

Train_easy , Valid_easy = split_it(EASY)

Train_hard , Valid_hard = split_it(HARD)

Totaldataset = ConcatDataset([EASY, HARD, BONUS])

train_dataset = ConcatDataset([Train_easy,Train_hard,Train_Bonusgreen,Train_Bonusred])
valid_dataset = ConcatDataset([Valid_easy,Valid_hard,Valid_Bonusred,Valid_Bonusgreen])


print(f"percentage split train= {len(train_dataset) / len(Totaldataset) * 100:.2f}" )
print(f"percentage split valid= {len(valid_dataset) / len(Totaldataset) * 100:.2f}" )



print(f"Total Samples: {len(Totaldataset)}")
print(f"Train Samples: {len(train_dataset)}")
print(f"Valid  Samples: {len(valid_dataset)}")

In [ ]:
# Custom collate function to handle variable length labels
def pad_collate(batch):
    images, labels, label_lengths,_ = zip(*batch)
    images = torch.stack(images, 0)

    # Pad all labels to the max length in this batch (required for tensor stacking)
    max_len = max([len(l) for l in labels])
    padded_labels = torch.zeros(len(labels), max_len, dtype=torch.long)
    for i, l in enumerate(labels):
        padded_labels[i, :len(l)] = l

    return images, padded_labels, torch.tensor(label_lengths),_

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32 #dataset size small

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=pad_collate
)

val_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=pad_collate
)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

def levenshtein_distance(s1, s2):
  # edit distance
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)

    if len(s2) == 0:
        return len(s1)

    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row

    return previous_row[-1]

def calculate_metrics(preds, targets, encoder):
    """
    Decodes predictions and targets to strings, then calculates:
    1. Accuracy (Exact Word Match)
    2. Normalized Edit Distance (0.0 = Perfect, 1.0 = All Wrong)
    """
    # Preds: [Batch, SeqLen, Classes] -> Greedy Decode
    # Targets: [Batch, MaxLen] (Padded)

    batch_size = preds.size(0)
    correct_count = 0
    total_edit_distance = 0
    total_chars = 0

    # 1. Get Argmax Indices
    # Shape: [Batch, SeqLen]
    pred_indices = torch.argmax(preds, dim=2)

    for i in range(batch_size):
        # --- DECODE PREDICTION ---
        raw_pred = pred_indices[i].cpu().numpy()
        pred_text = ""
        prev_char = -1

        for char_idx in raw_pred:
            # CTC Logic: Merge repeats, drop blanks (0)
            if char_idx != 0 and char_idx != prev_char:
                pred_text += encoder.int2char[char_idx]
            prev_char = char_idx

        # --- DECODE TARGET ---
        # Strip padding (0s) from target
        target_seq = targets[i].cpu().numpy()
        target_text = "".join([encoder.int2char[c] for c in target_seq if c != 0])

        # --- METRICS ---
        if pred_text == target_text:
            correct_count += 1

        dist = levenshtein_distance(pred_text, target_text)
        total_edit_distance += dist
        total_chars += len(target_text)

    accuracy = correct_count / batch_size
    # Avoid div by zero
    norm_edit_dist = total_edit_distance / max(total_chars, 1)

    return accuracy, norm_edit_dist

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def __call__(self, val_loss, model):
        # Check if loss improved significantly
        if val_loss < (self.best_loss - self.min_delta):
            self.best_loss = val_loss
            self.counter = 0

        else:
            self.counter += 1
            print(f"    [EarlyStopping] {self.counter}/{self.patience} epochs without improvement.")
            if self.counter >= self.patience:
                self.early_stop = True

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt

PATIENCE = 5
LEARNING_RATE = 0.001
EPOCHS = 100
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- SETUP ---
# Initialize Model
model = CRNN(num_labels=len(VOCAB), hidden_size=256).to(DEVICE)

cnn_path = "/kaggle/input/ocrcnnvgg/pytorch/default/1/PretrainedFromClassifier.pth"

model.cnn.load_state_dict(torch.load(cnn_path), strict=False)

for param in model.cnn.parameters():
    
    param.requires_grad = False

print("Convmodule encoder loaded and frozen. Starting Stage 1 (Linear Probing)...")


criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE,weight_decay=1e-4)

# Early Stopping
early_stopper = EarlyStopping(patience=PATIENCE)

# Lists for Plotting later
history = {
    'train_loss': [], 'val_loss': [],
    'val_acc': [], 'val_edit_dist': []
}

# --- TRAINING LOOP ---
print(f"Starting Training on {DEVICE}...")

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    # ==========================
    # 1. TRAINING PHASE
    # ==========================
    if epoch == 5:
        print("\n>>> SWITCHING TO STAGE 2: Unfreezing Encoder & Lowering LR <<<")

        # 1. Unfreeze everything
        for param in model.parameters():
            param.requires_grad = True

        new_lr = LEARNING_RATE / 10 
        optimizer = optim.AdamW(model.parameters(), lr=new_lr, weight_decay=1e-4)
            
    model.train()
    train_loss_accum = 0

    for batch_idx, (images, targets, target_lengths,_) in enumerate(train_loader):
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        target_lengths = target_lengths.to(DEVICE)

        # Forward
        preds = model(images)

        # Prepare for CTC Loss
        log_probs = F.log_softmax(preds, dim=2)
        log_probs_permuted = log_probs.permute(1, 0, 2)
        input_lengths = torch.full((images.size(0),), preds.size(1), dtype=torch.long).to(DEVICE)

        # Loss & Backprop
        loss = criterion(log_probs_permuted, targets, input_lengths, target_lengths)
    
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        optimizer.step()

        train_loss_accum += loss.item()

    avg_train_loss = train_loss_accum / len(train_loader)

    # ==========================
    # 2. VALIDATION PHASE
    # ==========================
    model.eval()
    val_loss_accum = 0
    val_acc_accum = 0
    val_edit_accum = 0

    with torch.no_grad():
        for batch_idx, (images, targets, target_lengths,_) in enumerate(val_loader):
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            target_lengths = target_lengths.to(DEVICE)

            # Forward
            preds = model(images)

            # Loss Calculation
            log_probs = F.log_softmax(preds, dim=2)
            log_probs_permuted = log_probs.permute(1, 0, 2)
            input_lengths = torch.full((images.size(0),), preds.size(1), dtype=torch.long).to(DEVICE)

            loss = criterion(log_probs_permuted, targets, input_lengths, target_lengths)
            val_loss_accum += loss.item()

            # Metrics Calculation
            acc, edit_dist = calculate_metrics(preds, targets, encoder)
            val_acc_accum += acc
            val_edit_accum += edit_dist



    avg_val_loss = val_loss_accum / len(val_loader)
    avg_val_acc = val_acc_accum / len(val_loader)
    avg_val_edit = val_edit_accum / len(val_loader)

    # --- LOGGING ---
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val Loss:   {avg_val_loss:.4f} | Acc: {avg_val_acc*100:.2f}% | Edit Dist: {avg_val_edit:.4f}")

    # Update History
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_acc'].append(avg_val_acc)
    history['val_edit_dist'].append(avg_val_edit)

    # --- EARLY STOPPING CHECK ---
    early_stopper(avg_val_loss, model)

    if early_stopper.early_stop:
        print(f"\n[Stopping] Early stopping triggered. Best Val Loss: {early_stopper.best_loss:.4f}")
        break

print("Training Complete.")


def plot_training_results(history):
    epochs = range(1, len(history['train_loss']) + 1)

    plt.figure(figsize=(15, 6))

    # PLOT 1: LOSS (As requested)
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'b-', label='Training Loss', linewidth=2)
    plt.plot(epochs, history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
    plt.title(f"patience of 5 (eary stopping)")
    plt.xlabel("Epochs")
    plt.ylabel("CTC Loss")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_acc'], 'g-', label='Validation Accuracy', linewidth=2)
    plt.plot(epochs, history['val_edit_dist'], 'orange', label='Edit Distance', linestyle='--')
    plt.title("Validation Performance")
    plt.xlabel("Epochs")
    plt.ylabel("Metric Score")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Run the plotter
plot_training_results(history)
torch.save(model.state_dict(),"Task2Genration.pth")

